In [1]:
from sklearn.cluster import KMeans
import sys
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.parameter import Parameter
import torch.nn as nn
from abc import ABC, abstractmethod
import torch.nn.functional as F
from pathlib import Path
root = Path.cwd()
if not (root / "modules").is_dir():
    REPO_ROOT = root.parent
sys.path.insert(0, str(REPO_ROOT))
from utilities.data_loader import load_embeddings
from utilities.utils import *
from tqdm import tqdm

In [2]:
class AE(nn.Module):
    def __init__(self, n_input, n_enc_1=256, n_enc_2=128, n_enc_3=64, n_z=32):
        super().__init__()

        # Encoder
        self.enc_1 = nn.Linear(n_input, n_enc_1)
        self.enc_2 = nn.Linear(n_enc_1, n_enc_2)
        self.enc_3 = nn.Linear(n_enc_2, n_enc_3)
        self.z_layer = nn.Linear(n_enc_3, n_z)

        # Decoder
        self.dec_1 = nn.Linear(n_z, n_enc_3)
        self.dec_2 = nn.Linear(n_enc_3, n_enc_2)
        self.dec_3 = nn.Linear(n_enc_2, n_enc_1)
        self.x_bar = nn.Linear(n_enc_1, n_input)

    def encode(self, x):
        h1 = F.relu(self.enc_1(x))
        h2 = F.relu(self.enc_2(h1))
        h3 = F.relu(self.enc_3(h2))
        z = self.z_layer(h3)
        return F.normalize(z, dim=1)

    def decode(self, z):
        d1 = F.relu(self.dec_1(z))
        d2 = F.relu(self.dec_2(d1))
        d3 = F.relu(self.dec_3(d2))
        return self.x_bar(d3)

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z


class IDECClusterer:
    def __init__(self, device: str):
        self.device = device
        self.training_history = []

    @staticmethod
    def target_distribution(q):
        weight = (q ** 2) / torch.sum(q, dim=0)
        return (weight.t() / torch.sum(weight, dim=1)).t()

    def soft_assign(self, z, centers):
        dist = torch.cdist(z, centers) ** 2
        q = 1.0 / (1.0 + dist)
        return q / q.sum(dim=1, keepdim=True)

    def fit_predict(self, embeddings: np.ndarray, k: int, true_labels=None, dataset_name="unknown"):

        torch.manual_seed(42)
        np.random.seed(42)
        self.training_history = []

        device = torch.device(self.device)

        X = torch.tensor(embeddings, dtype=torch.float32).to(device)

        latent_dim = 10

        dataset = TensorDataset(X)
        loader = DataLoader(dataset, batch_size=64, shuffle=False)

        model = AE(n_input=X.shape[1], n_z=latent_dim).to(device)

        model.cluster_centers = Parameter(
            torch.zeros(k, latent_dim, device=device)
        )

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        # pretrain autoencoder
        print("Pretraining autoencoder...")
        for epoch in tqdm(range(100)):
            total_loss = 0.0
            for (batch,) in loader:
                batch = batch.to(device)
                x_hat, _ = model(batch)
                loss = F.mse_loss(x_hat, batch)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            self.training_history.append({
                "stage": "pretrain",
                "epoch": epoch,
                "dataset": dataset_name,
                "reconstruction_loss": total_loss / len(loader)
            })

        # KMeans on latent space
        with torch.no_grad():
            z = model.encode(X).cpu().numpy()

        kmeans = KMeans(n_clusters=k, n_init=20, max_iter=300,
                        random_state=42)
        y_pred = kmeans.fit_predict(z)

        model.cluster_centers.data = torch.tensor(
            kmeans.cluster_centers_, dtype=torch.float32
        ).to(device)

        # Training loop for clustering
        print("Training clustering...")
        for epoch in tqdm(range(20)):

            # update target distribution
            with torch.no_grad():
                _, z = model(X)
                q_all = self.soft_assign(z, model.cluster_centers)
                q_all = torch.clamp(q_all, min=1e-10)

                p_all = self.target_distribution(q_all).detach()

                y_pred_new = q_all.argmax(1).cpu().numpy()
                delta = np.mean(y_pred != y_pred_new)
                y_pred = y_pred_new

            # convergence check
            if epoch > 30 and delta < 1e-3:
                self.training_history.append({
                    "stage": "cluster",
                    "epoch": epoch,
                    "dataset": dataset_name,
                    "delta": float(delta),
                    "converged": True
                })
                break

            # training loop
            total_loss = recon_loss_sum = kl_loss_sum = 0.0

            for (batch,) in loader:
                batch = batch.to(device)

                x_hat, z = model(batch)

                q = self.soft_assign(z, model.cluster_centers)
                q = torch.clamp(q, min=1e-10)

                p = self.target_distribution(q).detach()

                recon_loss = F.mse_loss(x_hat, batch)
                kl_loss = F.kl_div(q.log(), p, reduction="batchmean")

                total = recon_loss + 0.1 * kl_loss

                optimizer.zero_grad()
                total.backward()
                optimizer.step()

                total_loss += total.item()
                recon_loss_sum += recon_loss.item()
                kl_loss_sum += kl_loss.item()

            self.training_history.append({
                "stage": "cluster",
                "epoch": int(epoch),
                "total_loss": float(total_loss / len(loader)),
                "reconstruction_loss": float(recon_loss_sum / len(loader)),
                "kl_loss": float(kl_loss_sum / len(loader)),
                "delta": float(delta)

            })

            if true_labels is not None and epoch % 10 == 0:
                metrics = get_metrics(
                    X.cpu().numpy(),
                    true_labels,
                    y_pred
                )
                self.training_history.append({
                    "stage": "eval",
                    "epoch": epoch,
                    "dataset": dataset_name,
                    "NMI": metrics["NMI"],
                    "ARI": metrics["ARI"],
                    "ACC": metrics["Accuracy"]
                })

        # final prediction
        with torch.no_grad():
            _, z = model(X)
            q = self.soft_assign(z, model.cluster_centers)
            final_pred = q.argmax(1).cpu().numpy()

        # final metrics
        if true_labels is not None:
            metrics = get_metrics(X.cpu().numpy(), true_labels, final_pred)
            self.training_history.append({
                "stage": "final",
                "dataset": dataset_name,
                **metrics
            })

        return final_pred

In [3]:
def run_idec(X, n_clusters,device, y_true=None, dataset_name="unknown"):
    model = IDECClusterer(device)

    y_pred = model.fit_predict(
        embeddings=X,
        k=n_clusters,
        true_labels=y_true,
        dataset_name=dataset_name
    )

    return y_pred, model.training_history


def run_experiments(datasets_root, csv_path):

    datasets = list(Path(datasets_root).glob("*.npz"))
    
    history_dict = {}
    
    for dataset_path in datasets:

        dataset_name = dataset_path.stem.replace(
            "emb_", "").replace("_sbert", "")

        print(f"\nRunning dataset: {dataset_name}")

        X, y, texts = load_embeddings(dataset_path)

        X_reduced, pca = pca_by_variance(X)
        pca_components = pca.n_components_

        k_true = len(set(y))

        y_pred,history = run_idec(X_reduced, n_clusters=k_true, device=get_device(), y_true=y, dataset_name=dataset_name)

        history_dict[dataset_name] = history

        metrics = get_metrics(X_reduced, y, y_pred)

        plot_and_save_clusters(dataset=dataset_name,clusterer="IDEC", X=X_reduced,
                               y_pred=y_pred,
                               n_clusters=k_true,
                               number_of_components=pca_components)

        log_experiment(
            csv_path=csv_path,
            model_name="IDEC",
            dataset_name=dataset_name,
            n_rows=len(X_reduced),
            pca_components=pca_components,
            nmi=metrics["NMI"],
            ari=metrics["ARI"],
            acc=metrics["Accuracy"],
            purity=metrics["Purity"]
        )
        
    return history_dict


def parse_history(history):
    pretrain_epochs = []
    pretrain_loss = []

    cluster_epochs = []
    total_loss = []
    recon_loss = []
    kl_loss = []
    delta = []

    eval_epochs = []
    nmi = []
    ari = []
    acc = []

    for h in history:
        if h["stage"] == "pretrain":
            pretrain_epochs.append(h["epoch"])
            pretrain_loss.append(h["reconstruction_loss"])

        elif h["stage"] == "cluster":
            if "total_loss" in h:
                cluster_epochs.append(h["epoch"])
                total_loss.append(h["total_loss"])
                recon_loss.append(h["reconstruction_loss"])
                kl_loss.append(h["kl_loss"])
                delta.append(h["delta"])

        elif h["stage"] == "eval":
            eval_epochs.append(h["epoch"])
            nmi.append(h["NMI"])
            ari.append(h["ARI"])
            acc.append(h["ACC"])

    return {
        "pretrain": (pretrain_epochs, pretrain_loss),
        "cluster": (cluster_epochs, total_loss, recon_loss, kl_loss, delta),
        "eval": (eval_epochs, nmi, ari, acc)
    }


def plot_pretrain(data):
    epochs, loss = data

    plt.figure()
    plt.plot(epochs, loss)
    plt.xlabel("Epoch")
    plt.ylabel("Reconstruction Loss")
    plt.title("Pretraining Curve")
    plt.show()

def plot_cluster_losses(data):
    epochs, total, recon, kl, delta = data

    plt.figure()
    plt.plot(epochs, total, label="Total")
    plt.plot(epochs, recon, label="Reconstruction")
    plt.plot(epochs, kl, label="KL")
    plt.legend()
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Clustering Loss Breakdown")
    plt.show()

def plot_delta(data):
    epochs, _, _, _, delta = data

    plt.figure()
    plt.plot(epochs, delta)
    plt.xlabel("Epoch")
    plt.ylabel("Delta")
    plt.title("Cluster Assignment Change")
    plt.show()

def plot_metrics(data):
    epochs, nmi, ari, acc = data

    plt.figure()
    plt.plot(epochs, nmi, label="NMI")
    plt.plot(epochs, ari, label="ARI")
    plt.plot(epochs, acc, label="ACC")
    plt.legend()
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Clustering Metrics Over Time")
    plt.show()

def plot_kl_vs_acc(cluster_data, eval_data):
    c_epochs, _, _, kl, _ = cluster_data
    e_epochs, _, _, acc = eval_data

    # align epochs
    acc_map = dict(zip(e_epochs, acc))
    aligned_acc = [acc_map.get(e, None) for e in c_epochs]

    plt.figure()
    plt.plot(kl, aligned_acc, marker='o')
    plt.xlabel("KL Loss")
    plt.ylabel("Accuracy")
    plt.title("KL vs Accuracy Relationship")
    plt.show()

In [ ]:
history = run_experiments(
        datasets_root="../embeddings",
        csv_path="../outputs/results.csv"
    )